# GLM Power Creep Analysis

This notebook rebuilds the attack-focused GLM study using the curated dataset under `data/data_clean.csv`. The layout mirrors the lightweight notebooks in `analysis/`, while the modeling choices follow `tychtjan_analysis/GLM_context.md` and the exploratory work in `tychtjan_analysis/glm_analysis.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_squared_error

import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

from IPython.display import display

sns.set_theme(style="whitegrid", context="talk")
pd.options.display.float_format = "{:.2f}".format

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "data_clean.csv"
if not DATA_PATH.exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_PATH = PROJECT_ROOT / "data" / "data_clean.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Cannot locate data_clean.csv from {PROJECT_ROOT}. Check the repo layout."
    )

SyntaxError: unterminated string literal (detected at line 23) (1681579224.py, line 23)

### Load and preview the cleaned Pokémon dataset

In [ ]:
raw_df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(raw_df):,} | Columns: {raw_df.shape[1]}")
raw_df.head()

### Feature engineering pipeline

We reproduce the engineered fields described in `GLM_context.md`: composite stats, mechanic complexity flags, resistance PCA, and lightly transformed skewed columns. Type/id columns are mapped back to readable labels so categorical contrasts are easy to interpret.

In [ ]:
STAT_COLUMNS = ["hp", "attack", "defense", "sp_attack", "sp_defense", "speed"]
RESISTANCE_COLS = [col for col in raw_df.columns if col.startswith("against_")]
TYPE_ORDER = [
    "grass", "fire", "water", "bug", "normal", "poison", "electric", "ground",
    "fairy", "fighting", "psychic", "rock", "ghost", "ice", "dragon", "dark",
    "steel", "flying"
]
RARITY_LABELS = {0: "normal", 1: "sublegendary", 2: "legendary", 3: "mythical"}


def engineer_features(df: pd.DataFrame):
    out = df.copy().dropna(subset=["attack"]).reset_index(drop=True)
    out["gen"] = out["gen"].astype(int)
    # Composite offensive/defensive stats
    out["total_offense"] = out["attack"] + out["sp_attack"]
    out["total_defense"] = out["defense"] + out["sp_defense"]
    out["overall_stat_sum"] = out[STAT_COLUMNS].sum(axis=1)
    out["offense_ratio"] = out["total_offense"] / out["total_defense"].replace(0, np.nan)
    out["aggression_index"] = out["attack"] / out["overall_stat_sum"]
    out["offense_share"] = out["total_offense"] / out["overall_stat_sum"]
    # Skew-reducing transforms
    for col in ["weight_kg", "base_egg_steps", "capture_rate"]:
        out[f"log_{col}"] = np.log1p(out[col].clip(lower=0))
    for col in ["votes_first", "votes_top_6"]:
        out[f"sqrt_{col}"] = np.sqrt(out[col].clip(lower=0))
    # Mechanics flags
    out["mechanic_stack"] = out["has_mega_evolution"].fillna(0) + out["has_gigantamax"].fillna(0)
    out["long_evo_chain"] = (out["evo_length"].fillna(0) >= 3).astype(int)
    out["mechanic_complexity"] = out["mechanic_stack"] + out["long_evo_chain"]
    # Type / rarity labels for readable contrasts
    type_map = {idx: name.title() for idx, name in enumerate(TYPE_ORDER)}
    out["primary_type"] = out["class_primary"].map(type_map).fillna("Unknown")
    out["secondary_type"] = out["class_secondary"].map(type_map).fillna("None")
    out["rarity_label"] = out["rarity"].map(RARITY_LABELS).fillna("normal")
    # Resistance PCA (3 components explain ~45% of scaled variance)
    scaler = StandardScaler()
    resistance_scaled = scaler.fit_transform(out[RESISTANCE_COLS])
    pca = PCA(n_components=3, random_state=42)
    resistance_pcs = pca.fit_transform(resistance_scaled)
    for idx in range(3):
        out[f"resistance_pc{idx+1}"] = resistance_pcs[:, idx]
    meta = {
        "resistance_variance_ratio": pca.explained_variance_ratio_.round(4).tolist(),
        "resistance_cols": RESISTANCE_COLS,
    }
    # Clean up inf values produced by zero division
    out = out.replace({np.inf: np.nan, -np.inf: np.nan})
    return out, meta

In [ ]:
feature_df, resistance_meta = engineer_features(raw_df)
print("Feature rows:", len(feature_df))
print("Resistance PCA variance explained:", resistance_meta["resistance_variance_ratio"])
feature_df.head(3)

### Correlation diagnostics (Pearson / Spearman / Kendall)

Replicates the multi-metric correlation screen from the legacy SAN playbook to understand how offensive and defensive aggregates move with `attack`.

In [ ]:
corr_cols = [
    "attack", "total_offense", "total_defense", "overall_stat_sum",
    "hp", "defense", "sp_attack", "sp_defense", "speed"
]

corr_long = []
for method in ["pearson", "spearman", "kendall"]:
    corr_series = feature_df[corr_cols].corr(method=method)["attack"].drop(labels=["attack"])
    corr_long.append(pd.DataFrame({"metric": corr_series.index, "correlation": corr_series.values, "method": method}))

corr_table = pd.concat(corr_long).pivot(index="metric", columns="method", values="correlation")
print("Correlation vs. attack (sorted by Pearson strength):")
corr_table.sort_values(by="pearson", ascending=False)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(feature_df[corr_cols].corr(method="pearson"), annot=True, fmt=".2f", cmap="coolwarm", cbar_kws={"label": "Pearson r"})
plt.title("Core stat correlations")
plt.tight_layout()
plt.show()

### Generational aggregates and mechanic drift

Aggregate means mirror the summary table captured in the GLM context document, letting us verify whether later generations sustain higher offensive stats while mechanic complexity ebbs and flows.

In [ ]:
summary_cols = ["attack", "total_offense", "overall_stat_sum", "offense_ratio", "mechanic_complexity"]
gen_summary = feature_df.groupby("gen")[summary_cols].mean().round(2)
gen_summary

In [ ]:
melted = gen_summary.reset_index().melt(id_vars="gen", value_vars=["attack", "total_offense", "offense_ratio"], var_name="stat", value_name="value")
plt.figure(figsize=(10, 6))
sns.lineplot(data=melted, x="gen", y="value", hue="stat", marker="o")
plt.axvline(7, ls="--", color="grey", alpha=0.5)
plt.title("Generation-level offensive drift")
plt.ylabel("Mean value")
plt.tight_layout()
plt.show()

### ElasticNet feature screen

Following the penalized GLM idea in `GLM_context.md`, shrinkage helps surface which engineered predictors retain explanatory power once dummy expansions are included.

In [ ]:
enet_features = [
    "total_offense", "total_defense", "overall_stat_sum", "offense_ratio",
    "aggression_index", "offense_share", "hp", "defense", "sp_attack",
    "sp_defense", "speed", "mechanic_complexity", "num_abilities",
    "log_weight_kg", "log_base_egg_steps", "log_capture_rate",
    "sqrt_votes_first", "sqrt_votes_top_6", "resistance_pc1",
    "resistance_pc2", "resistance_pc3", "gen"
]

enet_df = feature_df.dropna(subset=enet_features + ["rarity_label", "primary_type", "secondary_type", "attack"])
X_enet = enet_df[enet_features]
X_enet = pd.get_dummies(
    pd.concat([X_enet, enet_df[["rarity_label", "primary_type", "secondary_type"]]], axis=1),
    columns=["rarity_label", "primary_type", "secondary_type"],
    drop_first=True
)
y_enet = enet_df["attack"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_enet)

enet = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], n_alphas=75, cv=5, random_state=42, max_iter=6000)
enet.fit(X_scaled, y_enet)

coef_series = pd.Series(enet.coef_, index=X_enet.columns)
non_zero = coef_series[coef_series.abs() > 1e-6].sort_values(key=np.abs, ascending=False).head(15)
print(f"Chosen alpha: {enet.alpha_:.3f} | l1_ratio: {enet.l1_ratio_:.2f}")
non_zero.to_frame(name="ElasticNet coef")

### Gaussian GLMs (with and without generational/mechanic terms)

We mirror the dual-model comparison from the context doc: first a stat-only baseline, then an augmented model that layers generation, resistance PCs, mechanics, and type controls.

In [ ]:
glm_cols = [
    "attack", "total_offense", "total_defense", "overall_stat_sum", "offense_ratio",
    "aggression_index", "offense_share", "sqrt_votes_first", "sqrt_votes_top_6",
    "log_weight_kg", "log_base_egg_steps", "log_capture_rate", "num_abilities",
    "mechanic_complexity", "resistance_pc1", "resistance_pc2", "resistance_pc3",
    "primary_type", "secondary_type", "rarity_label", "gen"
]

glm_df = feature_df.dropna(subset=glm_cols).copy()
glm_df["rarity_label"] = pd.Categorical(glm_df["rarity_label"], categories=["normal", "sublegendary", "legendary", "mythical"], ordered=True)
glm_df["primary_type"] = pd.Categorical(glm_df["primary_type"], categories=[name.title() for name in TYPE_ORDER] + ["Unknown"], ordered=False)
glm_df["secondary_type"] = pd.Categorical(glm_df["secondary_type"], categories=["None"] + [name.title() for name in TYPE_ORDER], ordered=False)

baseline_formula = "attack ~ total_offense + total_defense + overall_stat_sum + offense_ratio + aggression_index + offense_share + sqrt_votes_first + sqrt_votes_top_6 + log_weight_kg + log_base_egg_steps + log_capture_rate + num_abilities + C(rarity_label)"
rich_formula = baseline_formula + " + C(gen) + mechanic_complexity + resistance_pc1 + resistance_pc2 + resistance_pc3 + C(primary_type) + C(secondary_type)"

baseline_glm = smf.glm(formula=baseline_formula, data=glm_df, family=sm.families.Gaussian()).fit()
rich_glm = smf.glm(formula=rich_formula, data=glm_df, family=sm.families.Gaussian()).fit()

comparison = pd.DataFrame({
    "model": ["Baseline", "Generational + mechanics"],
    "AIC": [baseline_glm.aic, rich_glm.aic],
    "Deviance": [baseline_glm.deviance, rich_glm.deviance],
    "Pseudo_R2": [1 - baseline_glm.deviance / baseline_glm.null_deviance, 1 - rich_glm.deviance / rich_glm.null_deviance]
})
comparison

### Likelihood-ratio test for the generation block

Testing whether adding `C(gen)` + mechanics materially improves fit beyond the stat-only baseline.

In [ ]:
lr_stat = baseline_glm.deviance - rich_glm.deviance
df_diff = baseline_glm.df_resid - rich_glm.df_resid
p_value = stats.chi2.sf(lr_stat, df_diff)
print(f"LR stat = {lr_stat:.2f} on {int(df_diff)} df | p-value = {p_value:.3f}")

### Rolling generation cross-validation

Forward-chaining splits (train ≤ Gen k, test Gen k+1) emulate how well the GLM extrapolates into future mechanics, matching the `glm_analysis.ipynb` diagnostics.

In [ ]:
def rolling_generation_cv(df: pd.DataFrame, formula: str):
    results = []
    ordered_gens = sorted(df["gen"].unique())
    for idx in range(1, len(ordered_gens)):
        train_gens = ordered_gens[:idx]
        test_gen = ordered_gens[idx]
        train_df = df[df["gen"].isin(train_gens)]
        test_df = df[df["gen"] == test_gen]
        if len(test_df) < 5:
            continue
        model = smf.glm(formula=formula, data=train_df, family=sm.families.Gaussian()).fit()
        preds = model.predict(test_df)
        rmse = mean_squared_error(test_df["attack"], preds, squared=False)
        results.append({
            "train_gens": f"≤{train_gens[-1]}",
            "test_gen": test_gen,
            "rmse": rmse
        })
    return pd.DataFrame(results)

cv_results = rolling_generation_cv(glm_df, rich_formula)
cv_results

### Coefficient contributions (generational GLM)

Visualize which predictors push `attack` up/down after controlling for composites. Only the top absolute effects are shown to keep the chart readable.

In [ ]:
coef_table = (
    rich_glm.params.to_frame(name="coef")
    .join(rich_glm.bse.rename("stderr"))
    .drop(index="Intercept", errors="ignore")
)
coef_table["abs_coef"] = coef_table["coef"].abs()
top_coefs = coef_table.sort_values("abs_coef", ascending=False).head(20)
plt.figure(figsize=(8, 10))
sns.barplot(data=top_coefs, x="coef", y=top_coefs.index, orient="h", hue=(top_coefs["coef"] > 0), dodge=False, palette=["#ef5350", "#42a5f5"])
plt.xlabel("Coefficient (attack scale)")
plt.ylabel("Predictor")
plt.title("Top GLM coefficients")
plt.legend(title="Positive", loc="lower right")
plt.tight_layout()
plt.show()

### Poisson GLM on ability counts

A secondary GLM mirrors the context doc by modeling `num_abilities` with rarity, type, and generation terms to see whether newer species list fewer abilities once stats/mechanics are considered.

In [ ]:
poisson_formula = "num_abilities ~ total_offense + total_defense + mechanic_complexity + sqrt_votes_first + sqrt_votes_top_6 + C(rarity_label) + C(primary_type) + C(gen)"
poisson_df = feature_df.dropna(subset=["num_abilities", "total_offense", "total_defense", "mechanic_complexity", "sqrt_votes_first", "sqrt_votes_top_6", "rarity_label", "primary_type", "gen"]).copy()
poisson_df["rarity_label"] = pd.Categorical(poisson_df["rarity_label"], categories=["normal", "sublegendary", "legendary", "mythical"], ordered=True)
poisson_df["primary_type"] = pd.Categorical(poisson_df["primary_type"], categories=[name.title() for name in TYPE_ORDER] + ["Unknown"], ordered=False)

poisson_glm = smf.glm(formula=poisson_formula, data=poisson_df, family=sm.families.Poisson()).fit()
poisson_glm.summary2().tables[1].head(12)

### Next steps

* Run diagnostic plots (`rich_glm.plot_added()`, QQ, leverage) once the bug backlog is cleared.
* Extend the rolling CV to hold out multiple future generations (e.g., train ≤ Gen 5, test on 6–7).
* Re-run after filtering out legendries/mythicals to quantify how much of the creep is driven by rare species.